# Run ALE (Jupyter)

Computes ALE explanations for trained spatial models, without the plots of `XAI/run_ale.ipynb`. Start it from `XAI/scripts/`.

### Before You Run
- Build the `preles_c` extension (README, section 2).
- Put the trained models under `checkpoints/`, or set `CHECKPOINT_PATH`. The expected layout is
  `checkpoints/preles/full/station_<length>/logs_spatial_<model type>/outer_station_<site>/`.
- The PRELES calibrations for the models' training splits must exist under `Data/calibrations/`.
- Mixture-of-experts models are not covered here.


In [ ]:
# --- Imports and path setup ---
import os
import sys
import json
import glob
import warnings

import joblib

warnings.filterwarnings("ignore", message="Series.ravel is deprecated", category=FutureWarning)

PROJECT_ROOT = os.getenv("PROJECT_ROOT") or os.path.abspath(os.path.join(os.getcwd(), "../.."))
os.chdir(PROJECT_ROOT)   # data and calibration paths are relative to the repository root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from XAI.xai_utils.ale_registry import VARIANT_FN
from XAI.xai_utils.batch_utils import batch_explain


In [ ]:
# --- CONFIG (EDIT HERE) ---
variant = os.getenv("ALE_VARIANT", "main")      # {"main", "pairwise", "full"}
length = os.getenv("ALE_LENGTH", "len12")       # {"len3", "len6", "len9", "len12"}
prediction_types = ["naive", "bias", "parallel", "regularized", "residual"]

# False: PRELES outputs held fixed, ALE of the network alone.
# True ("main" and "full" only): each perturbed day is re-run through PRELES, then the network, so the
# curves show the full response to the weather. Hybrids only; naive and regularized do not use PRELES.
chain_process = os.getenv("ALE_CHAIN_PROCESS", "").lower() in {"1", "true", "yes"}
# "full" only: restrict to some feature pairs, e.g. [["PAR", "VPD"]]; None = all pairs
feature_pairs = [p.strip().split("_", 1) for p in os.getenv("ALE_PAIRS", "").split(",") if p.strip()] or None
if chain_process:
    prediction_types = [t for t in prediction_types if t in {"residual", "bias", "parallel"}]

grid_size = int(os.getenv("ALE_GRID_SIZE", 20)) # 10 coarser and quicker, 30 finer and slower
checkpoint_every = 1
save_intermediate = True
overwrite = False                               # False resumes from the per-type caches
pairwise_impute_empty_cells = True              # fills empty 2D bins with nearest-neighbour values;
                                                # can introduce combinations that do not occur in the data

# Paths
checkpoint_root = os.getenv("CHECKPOINT_PATH", os.path.join(PROJECT_ROOT, "checkpoints"))
logs_root = os.path.join(checkpoint_root, "preles", "full", f"station_{length}")
cv_config_path = os.path.join(PROJECT_ROOT, "configs", "CV_splits", "Paper_1_preles_ncvs",
                              f"CV_station_{length}_config.json")
save_dir_base = os.path.join(PROJECT_ROOT, "XAI", "results", "ALE", "paper1_spatial")
variant_dir = f"{variant}_chain" if chain_process else variant
root = os.path.join(save_dir_base, f"spatial_{length}", f"grid{grid_size}", variant_dir)
bundle_path = os.path.join(root, f"{variant}_bundle.joblib")

print("[CONFIG] variant:", variant)
print("[CONFIG] length:", length)
print("[CONFIG] prediction_types:", prediction_types)
print("[CONFIG] chain_process:", chain_process)
print("[CONFIG] feature_pairs:", feature_pairs or "all")
print("[CONFIG] logs_root:", logs_root)
print("[CONFIG] cv_config_path:", cv_config_path)
print("[CONFIG] output folder:", root)
print("[CONFIG] grid_size:", grid_size)
print("[CONFIG] overwrite:", overwrite)
if variant == "pairwise":
    print("[CONFIG] pairwise_impute_empty_cells:", pairwise_impute_empty_cells)
if chain_process and variant not in {"main", "full"}:
    raise ValueError("chain_process is only implemented for the 'main' and 'full' variants.")
if feature_pairs and variant != "full":
    raise ValueError("feature_pairs is only implemented for the 'full' variant.")


In [ ]:
# --- Load CV config ---
with open(cv_config_path, "r") as f:
    CV_CONFIG = json.load(f)


In [ ]:
# --- Preflight: warn if any prediction type has no matching logs ---
for pred_type in prediction_types:
    logs_glob = os.path.join(logs_root, f"logs_spatial_{pred_type}*")
    if not glob.glob(logs_glob):
        print(f"[WARNING] No log directories found for prediction type '{pred_type}' (pattern: {logs_glob})")


In [ ]:
# --- Run ALE batch ---
save_intermediate_dir = os.path.join(root, "intermediate") if save_intermediate else None
os.makedirs(save_intermediate_dir or root, exist_ok=True)

batch_kwargs = dict(
    grid_size=grid_size,
    save_intermediate_dir=save_intermediate_dir,
    checkpoint_every=checkpoint_every if checkpoint_every else 0,
    overwrite=bool(overwrite),
)
if variant == "pairwise":
    batch_kwargs["impute_empty_cells"] = pairwise_impute_empty_cells
if chain_process:
    batch_kwargs["chain_process"] = True
if feature_pairs:
    batch_kwargs["feature_pairs"] = feature_pairs

results = batch_explain(
    explanation_fn=VARIANT_FN[variant],
    prediction_types=prediction_types,
    logs_root=logs_root,
    cv_config=CV_CONFIG,
    save_dir=root,
    file_prefix=variant,
    target_list=["GPP", "ET"],
    use_process=True,
    **batch_kwargs,
)

# Normalisation constants per fold, to convert the z-scored ALE x-axes back to physical units.
norm_stats = results.pop("normalization_stats", {})
if norm_stats:
    with open(os.path.join(root, f"{variant}_normalization_stats.json"), "w") as f:
        json.dump(norm_stats, f, indent=2)

joblib.dump(results, bundle_path)

meta = dict(
    variant=variant, ncv_kind="spatial",
    length=length, grid_size=grid_size,
    prediction_types=prediction_types,
    chain_process=chain_process,
    feature_pairs=feature_pairs,
    logs_root=logs_root,
    save_dir_base=save_dir_base,
    cv_config_path=cv_config_path,
)
if variant == "full":
    meta["full_effect"] = "main+interaction"
with open(os.path.join(root, f"{variant}_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"[OK] Saved ALE to: {bundle_path}")
print(f"[OK] Per-type caches are under: {root}")
